In [0]:
import dataiku
import logging

# ==============================================================================
# Configuration
# ==============================================================================
IGNORE_TAG = "ignore_tag" 

# ==============================================================================
# Logging
# ==============================================================================
logging.basicConfig(
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

def disable_tagged_scenarios(target_tag: str):
    try:
        client = dataiku.api_client()
        project_keys = client.list_project_keys()
        logger.info(f"Found {len(project_keys)} projects to scan.")

        for project_key in project_keys:
            process_project(client, project_key, target_tag)
            
    except Exception as e:
        logger.error(f"Fatal error in main execution: {e}")

def process_project(client, project_key: str, target_tag: str):
    try:
        project = client.get_project(project_key)
        scenarios = project.list_scenarios()
        
        for scenario_item in scenarios:
            scenario_id = scenario_item['id']
            process_scenario(project, scenario_id, target_tag)
            
    except Exception as e:
        logger.warning(f"Failed to process project {project_key}: {e}")

def process_scenario(project, scenario_id: str, target_tag: str):
    try:
        scenario = project.get_scenario(scenario_id)
        settings = scenario.get_settings()
        
        # ----------------------------------------------------------------------
        # ATTRIBUTE RESOLUTION STRATEGY
        # ----------------------------------------------------------------------
        # The 'StepBasedScenarioSettings' object wraps the JSON definition.
        # We need to find the underlying dictionary to access 'tags'.
        
        settings_dict = None
        
        # Strategy A: Check for .settings (standard internal API)
        if hasattr(settings, 'settings') and isinstance(settings.settings, dict):
            settings_dict = settings.settings
        # Strategy B: Check for .raw_settings (sometimes present)
        elif hasattr(settings, 'raw_settings') and isinstance(settings.raw_settings, dict):
            settings_dict = settings.raw_settings
        # Strategy C: Check if the object itself acts as a dict (rare)
        elif isinstance(settings, dict):
            settings_dict = settings
        
        if settings_dict is None:
            # DEBUG: Log available attributes if we can't find the dict
            logger.warning(f"Scenario {scenario_id}: Could not resolve settings dict. "
                           f"Available attributes: {dir(settings)}")
            return

        # ----------------------------------------------------------------------
        # LOGIC APPLICATION
        # ----------------------------------------------------------------------
        
        # Ensure 'tags' list exists
        if 'tags' not in settings_dict:
            settings_dict['tags'] = []
            
        current_tags = settings_dict.get('tags',[])
        
        # Note: settings.active usually maps to settings_dict['active']
        # We use the dict directly to be safe.
        is_active = settings_dict.get('active', False)
        
        if target_tag in current_tags and is_active:
            logger.info(f"Disabling scenario: {scenario_id} in project {project.project_key}")
            
            # 1. Disable auto-trigger
            settings_dict['active'] = False
            # Update the wrapper if it has an active property to stay in sync
            if hasattr(settings, 'active'):
                settings.active = False
            
            # 2. Add 'auto_disabled' tag
            if "auto_disabled" not in current_tags:
                current_tags.append("auto_disabled")
                # No need to re-assign current_tags to settings_dict['tags'] 
                # because lists are mutable references, but we can do it to be explicit.
            
            # Save using the wrapper object
            # settings.save()

    except Exception as e:
        logger.warning(f"Error processing scenario {scenario_id}: {e}")

# ==============================================================================
# Main
# ==============================================================================
if __name__ == "__main__":
    disable_tagged_scenarios(IGNORE_TAG)